# Module 4: Random-Noise Federated Attack


## Stage Goal

This notebook isolates the random-noise poisoning baseline. It loads clean FL baselines from `clean_baselines.ipynb`, shows live poisoning-run output, and displays clean versus poisoned sample images. Run `train_v3.ipynb`, `train_surrogate.ipynb`, and `clean_baselines.ipynb` first so the target, surrogate, and clean-baseline artifacts exist.


## 1. Notebook Setup


In [5]:
from pathlib import Path
import sys

MODULE_DIR = Path.cwd()
if (MODULE_DIR / "4_Adversarial_FL").exists():
    MODULE_DIR = MODULE_DIR / "4_Adversarial_FL"
SRC_DIR = MODULE_DIR / "src"
for path in (MODULE_DIR.parent, SRC_DIR):
    path_text = str(path)
    if path_text not in sys.path:
        sys.path.insert(0, path_text)

from IPython.display import display
import matplotlib.pyplot as plt

from attack_notebook_utils import (
    clean_baselines_table,
    clean_result_from_artifact,
    clean_vs_attack_table,
    load_clean_baselines,
    make_poisoned_sample_batch,
    plot_algorithm_sweep,
    plot_clean_history,
    plot_clean_vs_attack,
    plot_poisoned_samples,
    plot_sweep_table,
    prepare_context,
    run_algorithm_sweep,
    run_attack_parameter_sweep,
    run_basic_attack,
)

plt.rcParams.update({"figure.dpi": 120, "axes.grid": False})


## 2. Configuration

Edit this cell to change data, FL, attack, or sweep settings. No YAML config is used.


In [6]:
BASE_FED_CONFIG = {
    "fraction_clients": 0.2,
    "num_clients": 100,
    "num_rounds": 30,
    "num_epochs": 5,
    "batch_size": 64,
    "global_stepsize": 1.0,
    "local_stepsize": 0.002,
    "criterion": "torch.nn.CrossEntropyLoss",
}

ALGORITHMS = {
    "FedAvg": {
        "fed_config": {**BASE_FED_CONFIG, "global_stepsize": 1.0},
        "optim_config": {},
    },
    "FedAdam": {
        "fed_config": {**BASE_FED_CONFIG, "global_stepsize": 0.01},
        "optim_config": {"beta1": 0.9, "beta2": 0.99, "epsilon": 1e-2},
    },
    "FedAdagrad": {
        "fed_config": {**BASE_FED_CONFIG, "global_stepsize": 0.01},
        "optim_config": {"epsilon": 1e-2},
    },
    "FedYogi": {
        "fed_config": {**BASE_FED_CONFIG, "global_stepsize": 0.01},
        "optim_config": {"beta1": 0.9, "beta2": 0.99, "epsilon": 1e-2},
    },
    "Scaffold": {
        "fed_config": {**BASE_FED_CONFIG, "global_stepsize": 1.0},
        "optim_config": {"c_init": 0.0},
    },
}

ATTACK_NAME = "Random Noise"

BASE_ATTACK = {
    "type": "random_noise",
    "target_label": 0,
    "poison_rate": 0.4,
    "step_size": 8 / 255,
    "criterion": "torch.nn.CrossEntropyLoss",
}

PARAMETER_SWEEP = [
    {"name": "noise 2/255", "attack": {"step_size": 2 / 255}},
    {"name": "noise 4/255", "attack": {"step_size": 4 / 255}},
    {"name": "noise 8/255", "attack": {"step_size": 8 / 255}},
    {"name": "noise 12/255", "attack": {"step_size": 12 / 255}},
]

CONFIG = {
    "attack_name": ATTACK_NAME,
    "quiet": False,
    "data_config": {
        "dataset_path": "./Data/Imagenette",
        "dataset_name": "Imagenette",
        "non_iid_per": 0,
        "eval_subset": "attack_eval",
        "validation_split": {"enabled": True, "seed": 42, "selection_fraction": 0.5},
    },
    "global_config": {"seed": 42, "device": "cuda"},
    "artifacts": {
        "dir": "artifacts",
        "target_checkpoint": "module4_v3_target.pt",
        "surrogate_checkpoint": "module4_surrogate.pt",
        "clean_baselines": "module4_clean_baselines.json",
    },
    "model_config": {
        "module": "model",
        "name": "MobileNetV3Transfer",
        "kwargs": {"pretrained": False, "num_classes": 10, "dropout": 0.1},
    },
    "algorithms": ALGORITHMS,
    "attack": {
        "seed": 42,
        "malicious_fraction": 0.2,
        "malicious_client_selection": {"mode": "seeded_random", "client_ids": []},
        "start_round": 1,
        "attack": BASE_ATTACK,
        "surrogate": {
            "checkpoint": "artifacts/module4_surrogate.pt",
            "pretrained": False,
            "num_classes": 10,
            "finetune_epochs": 0,
            "batch_size": 64,
            "learning_rate": 0.001,
            "weight_decay": 0.0,
            "freeze_backbone": False,
            "early_stop_patience": 0,
        },
    },
    "parameter_sweep": PARAMETER_SWEEP,
    "algorithm_sweep": ["FedAvg", "FedAdam", "FedAdagrad", "FedYogi", "Scaffold"],
}

context = prepare_context(CONFIG)
print(f"Device: {context['global_config']['device']}")
print(f"Target checkpoint: {context['target_checkpoint']}")
print(f"Surrogate checkpoint: {context['surrogate_checkpoint']}")
print(f"Clean baseline artifact: {context['artifact_dir'] / CONFIG['artifacts']['clean_baselines']}")


Device: cuda
Target checkpoint: /home/ahoop004/T3-Ciders-FL/4_Adversarial_FL/artifacts/module4_v3_target.pt
Surrogate checkpoint: /home/ahoop004/T3-Ciders-FL/4_Adversarial_FL/artifacts/module4_surrogate.pt
Clean baseline artifact: /home/ahoop004/T3-Ciders-FL/4_Adversarial_FL/artifacts/module4_clean_baselines.json


## 3. Load Clean Baseline Artifact

This loads the clean baselines saved by `clean_baselines.ipynb`; it does not run clean FL training.


In [7]:
clean_baselines = load_clean_baselines(context)
clean_results = clean_baselines["results"]
clean_fedavg = clean_result_from_artifact(clean_baselines, "FedAvg")

display(clean_baselines_table(clean_baselines))
plot_clean_history(clean_fedavg, title="Saved clean FedAvg baseline")


,algorithm,final_accuracy,final_loss,global_target_label_asr,num_rounds
0,FedAvg,97.71,0.0770,0.23,10
1,FedAdam,97.71,0.0770,0.23,10
2,FedAdagrad,97.66,0.0772,0.23,10
3,FedYogi,97.71,0.0770,0.23,10
4,Scaffold,97.66,0.0770,0.23,10


## 4. Basic FedAvg Attack

This runs FedAvg with the notebook attack enabled and compares it with the saved clean FedAvg artifact. Round-by-round poisoning output is shown below.


In [8]:
fedavg_attack = run_basic_attack(context, "FedAvg")

display(clean_vs_attack_table(clean_fedavg, fedavg_attack))
print(f"Poisoned examples: {fedavg_attack['poisoned_examples']}")
print(f"Surrogate poison success: {fedavg_attack['surrogate_poison_success_rate']:.2f}%")
print(f"Global target-label ASR: {fedavg_attack['global_target_label_asr']:.2f}%")
plot_clean_vs_attack(
    clean_fedavg,
    fedavg_attack,
    title=f"FedAvg: clean artifact vs {CONFIG['attack_name']}",
    attack_start_round=CONFIG["attack"]["start_round"],
)



Preparing data with Dirichlet partitioner (aligned with Module 2)
Loading cached client data from cache/client_data_9b5ab701ecf4bad27b1ab2a8e0a01a5f.pkl


Clients successfully initialised

Communication Round: 1


[Server] Round 1/30 → selected 20 clients, 5 local epoch(s)


	client_update completed
	server_update completed
	Loss: 0.0760   Accuracy: 97.76%

Communication Round: 2


	Server Loss: 0.0760   Accuracy: 97.76%
[Server] Round 2/30 → selected 20 clients, 5 local epoch(s)


...

	Server Loss: 0.1193   Accuracy: 96.38%
[Server] Round 30/30 → selected 20 clients, 5 local epoch(s)


	client_update completed
	server_update completed
	Loss: 0.1829   Accuracy: 94.14%


	Server Loss: 0.1829   Accuracy: 94.14%


,algorithm,run,final_clean_accuracy,final_attacked_accuracy,accuracy_drop,global_target_label_asr,surrogate_poison_success_rate,poisoned_examples,malicious_fraction,num_rounds
0,FedAvg,Random Noise,97.71,94.14,3.57,1.36,11.81,21647,0.2,30


Poisoned examples: 21647
Surrogate poison success: 11.81%
Global target-label ASR: 1.36%


## 5. Attack Parameter Sweep

This keeps FedAvg fixed and changes only the attack settings listed in `PARAMETER_SWEEP`.


In [ ]:
# parameter_sweep = run_attack_parameter_sweep(context, clean_fedavg, "FedAvg")

# display(parameter_sweep["table"])
# plot_sweep_table(
#     parameter_sweep["table"],
#     title=f"FedAvg {CONFIG['attack_name']} parameter sweep",
# )


## 6. Algorithm Sweep

This compares attacked FL algorithms against the clean algorithm baselines loaded from the shared artifact.


In [ ]:
# algorithm_sweep = run_algorithm_sweep(context, clean_results=clean_results)

# display(algorithm_sweep["table"])
# plot_algorithm_sweep(
#     algorithm_sweep["table"],
#     title=f"Algorithm sweep under {CONFIG['attack_name']}",
# )


## 7. Final Plots and Poisoned Samples


In [10]:
display(clean_vs_attack_table(clean_fedavg, fedavg_attack))
plot_clean_vs_attack(
    clean_fedavg,
    fedavg_attack,
    title=f"FedAvg final view: clean artifact vs {CONFIG['attack_name']}",
    attack_start_round=CONFIG["attack"]["start_round"],
)

# plot_sweep_table(
#     parameter_sweep["table"],
#     title=f"Final parameter sweep: {CONFIG['attack_name']}",
# )
# plot_algorithm_sweep(
#     algorithm_sweep["table"],
#     title=f"Final algorithm sweep: {CONFIG['attack_name']}",
# )

poisoned_samples = make_poisoned_sample_batch(context, count=6)
plot_poisoned_samples(
    poisoned_samples,
    title=f"{CONFIG['attack_name']} poisoned samples from the held-out attack split",
)


,algorithm,run,final_clean_accuracy,final_attacked_accuracy,accuracy_drop,global_target_label_asr,surrogate_poison_success_rate,poisoned_examples,malicious_fraction,num_rounds
0,FedAvg,Random Noise,97.71,94.14,3.57,1.36,11.81,21647,0.2,30
